## HyperSense — Dataset Extraction and Harmonization
### Objective
#### Prepare a harmonized adult hypertension dataset from:

- Ghana DHS 2014 Men (MR)
- Ghana DHS 2014 Women (IR)
- Benin DHS 2017–18 Men (MR)
- Benin DHS 2017–18 Women (IR); for development of a hypertension risk prediction model.

**Special Consideration:**
Benin DHS 2017–18 did not collect anthropometric measurements for men. Ghana DHS 2014 male BMI is available within the Household Recode (HR) dataset and must be merged into the Men's Recode (MR) dataset before extraction.

**Workflow:**
1. Verify Ghana male anthropometry variables.
2. Merge Ghana HR BMI into Ghana MR.
3. Extract required variables from Ghana Men.
4. Extract required variables from Ghana Women.
5. Extract required variables from Benin Men.
6. Extract required variables from Benin Women.
7. Standardize variable names and coding.
8. Generate harmonized datasets.
9. Save cleaned outputs for EDA and modeling.

**Key thing to note:** 
The datasets are being simultaenously explored with *IBM SPSS Statistics 25* for to increase merging and EDA speed

### Step 1 - Ghana Men BMI Recovery

In [1]:
import pandas as pd
import pyreadstat 

In [8]:
hr_df, meta = pyreadstat.read_sav(r"C:\Users\USER\Desktop\HyperSense\data\Ghana 2014\GHHR72SV\GHHR72FL.SAV")
print(hr_df.shape)

(11835, 3028)


In [26]:
print(hr_df.columns[:30].tolist())

['HHID', 'HV000', 'HV001', 'HV002', 'HV003', 'HV004', 'HV005', 'HV006', 'HV007', 'HV008', 'HV009', 'HV010', 'HV011', 'HV012', 'HV013', 'HV014', 'HV015', 'HV016', 'HV017', 'HV018', 'HV019', 'HV020', 'HV021', 'HV022', 'HV023', 'HV024', 'HV025', 'HV026', 'HV027', 'HV028']


In [37]:
# ------------------------------------------------------------------
# STEP 1A: Inspect Ghana HR anthropometry variables
# ------------------------------------------------------------------

In [41]:
# BMI
print([c for c in hr_df.columns if "HB40" in c])

['HB40$1', 'HB40$2', 'HB40$3', 'HB40$4', 'HB40$5', 'HB40$6', 'HB40$7', 'HB40$8']


In [30]:
# Age of household members
print([c for c in hr_df.columns if "HV105" in c])

['HV105$01', 'HV105$02', 'HV105$03', 'HV105$04', 'HV105$05', 'HV105$06', 'HV105$07', 'HV105$08', 'HV105$09', 'HV105$10', 'HV105$11', 'HV105$12', 'HV105$13', 'HV105$14', 'HV105$15', 'HV105$16', 'HV105$17', 'HV105$18', 'HV105$19', 'HV105$20', 'HV105$21', 'HV105$22', 'HV105$23', 'HV105$24', 'HV105$25']


In [32]:
# Sex of household member
print([c for c in hr_df.columns if "HV104" in c])

['HV104$01', 'HV104$02', 'HV104$03', 'HV104$04', 'HV104$05', 'HV104$06', 'HV104$07', 'HV104$08', 'HV104$09', 'HV104$10', 'HV104$11', 'HV104$12', 'HV104$13', 'HV104$14', 'HV104$15', 'HV104$16', 'HV104$17', 'HV104$18', 'HV104$19', 'HV104$20', 'HV104$21', 'HV104$22', 'HV104$23', 'HV104$24', 'HV104$25']


In [20]:
# Find all BMI-related columns in the Ghana Household Recode file
# Goal: determine how many household roster positions have BMI measurements

hb40_cols = [c for c in hr_df.columns if "HB40" in c]
print(sorted(hb40_cols))

['HB40$1', 'HB40$2', 'HB40$3', 'HB40$4', 'HB40$5', 'HB40$6', 'HB40$7', 'HB40$8']


In [53]:
# Find all HB41 (Rohrer's index) columns
# Goal: identify whether HB41 contains additional anthropometric measures

hb41_cols = [c for c in hr_df.columns if "HB41" in c]
print(sorted(hb41_cols))

['HB41$1', 'HB41$2', 'HB41$3', 'HB41$4', 'HB41$5', 'HB41$6', 'HB41$7', 'HB41$8']


In [52]:
# Confirm that HVIDX (Line number) exists for all roster positions

print(sorted([c for c in hr_df.columns if "HVIDX" in c]))

['HVIDX$01', 'HVIDX$02', 'HVIDX$03', 'HVIDX$04', 'HVIDX$05', 'HVIDX$06', 'HVIDX$07', 'HVIDX$08', 'HVIDX$09', 'HVIDX$10', 'HVIDX$11', 'HVIDX$12', 'HVIDX$13', 'HVIDX$14', 'HVIDX$15', 'HVIDX$16', 'HVIDX$17', 'HVIDX$18', 'HVIDX$19', 'HVIDX$20', 'HVIDX$21', 'HVIDX$22', 'HVIDX$23', 'HVIDX$24', 'HVIDX$25']


In [40]:
# Check how many BMI values exist in each roster slot

for i in range(1, 9):
    col = f"HB40${i}"
    print(col, hr_df[col].notna().sum())

HB40$1 3551
HB40$2 659
HB40$3 164
HB40$4 32
HB40$5 8
HB40$6 1
HB40$7 1
HB40$8 1


In [56]:
# ------------------------------------------------------------------
# STEP 1B: Identify merge keys between HR and MR datasets
# ------------------------------------------------------------------

### BMI Structure Verification - Ghana Household Recode

At this point, I have been able to confirm that the Ghana 2014 Household Recode (HR) dataset contains BMI measurements (`HB40`) and household roster line numbers (`HVIDX`).

However, before attempting any merge with the Men's Recode (MR) dataset, there's a need to understand exactly how DHS stored these BMI values.

The key question is:

> Does `HB40$1` belong to household member `HVIDX$1`, `HB40$2` belong to `HVIDX$2`, and so on?

If true, then we can safely reshape the household BMI data into a person-level table and merge it directly to the Men's Recode using:

- `HV001 ↔ MV001` (Cluster Number)
- `HV002 ↔ MV002` (Household Number)
- `HVIDX ↔ MV003` (Respondent Line Number)

Initial inspection already suggests this is the case:

- BMI values exist in `HB40$1`–`HB40$8`.
- Corresponding household member line numbers exist in `HVIDX$1`–`HVIDX$8`.
- BMI appears to be stored in DHS standard format (BMI × 100). Example: `1703 → 17.03 BMI`.

The inspection below is therefore a sanity check to verify the roster structure before reshaping and merging.

In [45]:
# First of all, inspect household containing BMI measurements

cols = [
    "HV001", "HV002",
    "HVIDX$01", "HV104$01", "HV105$01", "HB40$1",
    "HVIDX$02", "HV104$02", "HV105$02", "HB40$2"
]

hr_df.loc[hr_df["HB40$1"].notna(), cols].head()

,HV001,HV002,HVIDX$01,HV104$01,HV105$01,HB40$1,HVIDX$02,HV104$02,HV105$02,HB40$2
0,1.0,1.0,1.0,1.0,50.0,1703.0,2.0,2.0,42.0,NaN
1,1.0,2.0,1.0,1.0,72.0,1964.0,2.0,2.0,60.0,1909.0
2,1.0,3.0,1.0,1.0,27.0,2048.0,2.0,2.0,25.0,NaN
4,1.0,6.0,1.0,1.0,24.0,2108.0,2.0,2.0,19.0,NaN
9,1.0,11.0,1.0,1.0,40.0,1972.0,NaN,NaN,NaN,NaN


In [50]:
# Count households with members in roster positions 9–25
for i in range(9, 26):
    idx_col = f"HVIDX${i:02d}"

    if idx_col in hr_df.columns:
        print(idx_col, hr_df[idx_col].notna().sum())

HVIDX$09 499
HVIDX$10 286
HVIDX$11 170
HVIDX$12 125
HVIDX$13 77
HVIDX$14 57
HVIDX$15 36
HVIDX$16 23
HVIDX$17 18
HVIDX$18 11
HVIDX$19 8
HVIDX$20 4
HVIDX$21 3
HVIDX$22 1
HVIDX$23 1
HVIDX$24 1
HVIDX$25 1


In [51]:
# Confirm maximum roster position recorded
hvidx_cols = [c for c in hr_df.columns if "HVIDX" in c]
hr_df[hvidx_cols].max().max()

np.float64(25.0)

**Conclusion:** Household roster positions extend to 25 members, but BMI measurements (`HB40`) are only populated for slots 1–8, indicating that anthropometry was collected for a subset of eligible household members rather than the entire household roster.

In [57]:
# ------------------------------------------------------------------
# STEP 1C: Recover Ghana male BMI from household roster
# ------------------------------------------------------------------

##### Step 1C - Construct Ghana Male BMI Lookup Table

Objective:

Transform the household-level anthropometry variables (HB40$1–HB40$8) into a person-level BMI lookup table containing:

- HV001 (Cluster Number)
- HV002 (Household Number)
- HVIDX (Household Member Line Number)
- BMI

This lookup table will later be merged into the Ghana Men's Recode dataset using:

- HV001 ↔ MV001
- HV002 ↔ MV002
- HVIDX ↔ MV003

Expected output:

HV001 | HV002 | HVIDX | BMI

In [58]:
# Collect BMI slots available in the household file

bmi_slots = []

for i in range(1, 9):
    bmi_slots.append(i)

bmi_slots

[1, 2, 3, 4, 5, 6, 7, 8]

In [63]:
# Create empty list to store BMI records

bmi_records = []

# Loop through the 8 BMI slots in the household roster
for i in range(1, 9):

    bmi_col = f"HB40${i}"
    idx_col = f"HVIDX${i:02d}"

    # Keep only rows where BMI exists
    filtered_result = hr_df.loc[
        hr_df[bmi_col].notna(),
        ["HV001", "HV002", idx_col, bmi_col]
    ].copy()
    
    temp = filtered_result
    
    # Standardize column names
    temp.columns = ["HV001", "HV002", "HVIDX", "BMI"]

    bmi_records.append(temp)

# Combine all slots into one long table
bmi_lookup = pd.concat(bmi_records, ignore_index=True)

print(bmi_lookup.shape)
bmi_lookup.head()

(4417, 4)


,HV001,HV002,HVIDX,BMI
0,1.0,1.0,1.0,1703.0
1,1.0,2.0,1.0,1964.0
2,1.0,3.0,1.0,2048.0
3,1.0,6.0,1.0,2108.0
4,1.0,11.0,1.0,1972.0


In [66]:
# Last sanity check pre-merge: Check whether any person appears more than once

duplicates = bmi_lookup.duplicated(
    subset=["HV001", "HV002", "HVIDX"]
).sum()

print("Duplicate keys:", duplicates)

Duplicate keys: 0


In [67]:
# ------------------------------------------------------------------
# STEP 1D: Merge recovered BMI into Ghana Men's Recode
# ------------------------------------------------------------------

In [82]:
# Load Ghana Men Recode (MR)

gm_df, meta = pyreadstat.read_sav(
    r"C:\Users\USER\Desktop\HyperSense\data\Ghana 2014\GHMR71SV\GHMR71FL.SAV"
)

print(gm_df.shape)

(4388, 655)


In [83]:
# Verify merge identifiers exist

gm_df[["MV001", "MV002", "MV003"]].head()

,MV001,MV002,MV003
0,1.0,1.0,1.0
1,1.0,3.0,1.0
2,1.0,6.0,1.0
3,1.0,11.0,1.0
4,1.0,19.0,1.0


In [84]:
# Create merge-compatible keys

bmi_lookup = bmi_lookup.rename(
    columns={
        "HV001": "MV001",
        "HV002": "MV002",
        "HVIDX": "MV003"
    }
)

bmi_lookup.head()

,MV001,MV002,MV003,BMI
0,1.0,1.0,1.0,1703.0
1,1.0,2.0,1.0,1964.0
2,1.0,3.0,1.0,2048.0
3,1.0,6.0,1.0,2108.0
4,1.0,11.0,1.0,1972.0


In [85]:
# Merge recovered BMI into Ghana Men dataset

gm_df = gm_df.merge(
    bmi_lookup,
    on=["MV001", "MV002", "MV003"],
    how="left"
)

print(gm_df.shape)

(4388, 656)


In [87]:
# How many men received a BMI value?

print("Total men:", len(gm_df))
print("BMI available:", gm_df["BMI"].notna().sum())
print("BMI missing:", gm_df["BMI"].isna().sum())

Total men: 4388
BMI available: 3109
BMI missing: 1279


In [88]:
# Quick look at recovered BMI values

gm_df[["MV001", "MV002", "MV003", "BMI"]].head(10)

,MV001,MV002,MV003,BMI
0,1.0,1.0,1.0,1703.0
1,1.0,3.0,1.0,2048.0
2,1.0,6.0,1.0,2108.0
3,1.0,11.0,1.0,1972.0
4,1.0,19.0,1.0,2300.0
5,1.0,19.0,3.0,NaN
6,1.0,23.0,1.0,2794.0
7,1.0,26.0,1.0,1995.0
8,1.0,27.0,3.0,NaN
9,1.0,30.0,1.0,2074.0


### Step 2 — Extract Required Variables from Ghana Men (MR)

Now that BMI has been successfully recovered and merged into the Ghana Men's Recode dataset, the next step is to extract only the variables required for harmonization and modeling.

The goal is to reduce the full MR dataset to a Ghana Men analysis dataset containing demographic, behavioral, anthropometric, blood pressure, and survey design variables... thereafter doing the same for the three other datasets: Ghana Women, Benin Men, and Benin Women!

In [90]:
# Show all Ghana Men columns

print(gm_df.columns.tolist())

['MCASEID', 'MV000', 'MV001', 'MV002', 'MV003', 'MV004', 'MV005', 'MV006', 'MV007', 'MV008', 'MV009', 'MV010', 'MV011', 'MV012', 'MV013', 'MV014', 'MV015', 'MV016', 'MV021', 'MV022', 'MV023', 'MV024', 'MV025', 'MV026', 'MV027', 'MV028', 'MV029', 'MV030', 'MV031', 'MV032', 'MV034$1', 'MV034$2', 'MV034$3', 'MV034$4', 'MV034$5', 'MV034$6', 'MV034$7', 'MV034$8', 'MV034A$1', 'MV034A$2', 'MV034A$3', 'MV034A$4', 'MV034A$5', 'MV034A$6', 'MV034A$7', 'MV034A$8', 'MV034B$1', 'MV034B$2', 'MV034B$3', 'MV034B$4', 'MV034B$5', 'MV034B$6', 'MV034B$7', 'MV034B$8', 'MV035', 'MV801', 'MV802', 'MV803', 'MV101', 'MV102', 'MV103', 'MV104', 'MV105', 'MV106', 'MV107', 'MV130', 'MV131', 'MV133', 'MV134', 'MV135', 'MV136', 'MV138', 'MV149', 'MV150', 'MV151', 'MV152', 'MV155', 'MV156', 'MV157', 'MV158', 'MV159', 'MV167', 'MV168', 'MV190', 'MV191', 'MV201', 'MV202', 'MV203', 'MV204', 'MV205', 'MV206', 'MV207', 'MV212', 'MV213', 'MV217', 'MV218', 'MV225', 'MV245', 'MV246', 'MV247', 'MV248', 'MV249', 'MV250', 'MV251